In [1]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from scipy import ndimage
import pandas as pd
import os
import math

#Pending 006 , 007, 008
image_path = r'XY_clean\xy017.jpg'
image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
scene_coordinate_system = -16.41414 #Variable depending of the image
width_pixels = image.shape[1]  
# 4.5 mm 
escala_mm_px = 4.5 / width_pixels


AttributeError: 'NoneType' object has no attribute 'shape'

In [1]:
# Definir umbrales
max_area_threshold = 10000 #10000
min_area_threshold = 0    # Umbral

# 1. Suavizar la imagen para reducir ruido
blur = cv2.GaussianBlur(image, (5, 5), 0) #23,23,(<008) 5,5(blackers-009)

# 2. Binarizar usando umbral de Otsu
_, binary = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
# invertir la máscara para que las burbujas sean foreground
binary = cv2.bitwise_not(binary)

# 3. Operaciones morfológicas para limpiar pequeñas manchas y separar un poco las burbujas
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3)) # 15 15, 3,3(blackers - 009)
binary_opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=5)

# 4.Transformada de distancia
dist_transform = cv2.distanceTransform(binary_opened, cv2.DIST_L2, 5)

# 5. Encontrar los picos locales en la transformada de distancia
coordinates = peak_local_max(dist_transform, min_distance=10)

# Coordenadas a una máscara booleana
local_max = np.zeros(dist_transform.shape, dtype=bool)
local_max[tuple(coordinates.T)] = True

# Etiquetar los picos para usarlos como marcadores en watershed
markers, _ = ndimage.label(local_max)

# 6.  la transformada de distancia negativa (para encontrar valles)
labels = watershed(-dist_transform, markers, mask=binary_opened)

# 7.Resultado
num_labels = labels.max()
np.random.seed(42)
colors = np.random.randint(0, 255, (num_labels+1, 3), dtype=np.uint8)

colored_result = np.zeros((labels.shape[0], labels.shape[1], 3), dtype=np.uint8)
valid_count = 0

for lbl in range(1, num_labels+1):
    mask = (labels == lbl).astype(np.uint8)
    area = cv2.countNonZero(mask)
    
    # Filtrar por área 
    if min_area_threshold <= area <= max_area_threshold:
        colored_result[labels == lbl] = colors[lbl]
        valid_count += 1

plt.figure(figsize=(12,12))
plt.imshow(cv2.cvtColor(binary, cv2.COLOR_BGR2RGB))
plt.title(f"Burbujas detectadas: {valid_count}")
plt.axis('off')
plt.show()
#Por tamaño, patrones colores
#Porcentaje de vacio, ocupado
#Modificar/Obtimizar segmentacion de las burbujas agrupadas

NameError: name 'cv2' is not defined

In [ ]:

image_name = os.path.splitext(os.path.basename(image_path))[0]

result_data = []
for lbl in range(1, num_labels+1):
    mask = (labels == lbl).astype(np.uint8)
    area_px = cv2.countNonZero(mask)

    if not (min_area_threshold <= area_px <= max_area_threshold):
        # Si no cumple el filtro, omitir esta burbuja
        continue

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) > 0:
        perimeter_px = cv2.arcLength(contours[0], True)
    else:
        perimeter_px = 0.0
    # Conversiones a mm
    diametro_px = math.sqrt((4 * area_px) / math.pi)
    area_mm2 = area_px * (escala_mm_px**2)
    perimetro_mm = perimeter_px * escala_mm_px
    diametro_mm = diametro_px * escala_mm_px

    result_data.append({
        'Imagen': image_name,
        'Scene_coordinate_system': scene_coordinate_system,
        'Area_px': area_px,
        'Area_mm2': area_mm2,
        'Perimetro_px': perimeter_px,
        'Perimetro_mm': perimetro_mm,
        'Diametro_px': diametro_px,
        'Diametro_mm': diametro_mm,
        'Escala_mm_por_px': escala_mm_px
    })
    valid_count += 1

In [267]:
import os

csv_filename = "Bubbles_Results_V2.csv"  # CSV global

df = pd.DataFrame(result_data)

if not os.path.exists(csv_filename):
    
    df.to_csv(csv_filename, index=False)
else:
    
    df.to_csv(csv_filename, mode='a', header=False, index=False)

